# Coaching-proof verification: a seeded sweep

**Tian Liang (tl356) · COMSCI/ECON 206 · PS1**

**Research question.** Which verification mechanisms remain effective when an adversary can
script the reporting party's responses?

This notebook re-implements the mechanism behind the interactive game
(https://dku-comsci-econ206-2026-tian-liang.static.hf.space/index.html) in Python, so that the
same model can be swept over parameters rather than played by hand.

Sequence: **predict → run → change one assumption → check → interpret**.

**Predictions, written before running.**
1. For the record check, the miss rate should track `1 − reporting rate`, because a payee is
   flagged with probability equal to the reporting rate.
2. For the testimony check, the miss rate should rise with the script rate and reach 1 when every
   approach is scripted — a scripted answer never conflicts, whatever the attention level.
3. Removing habituation should improve the testimony check at low script rates but not at high
   ones. If that holds, habituation and authorship are separate failure channels, and an
   interface-level remedy cannot address the second.

**Evidence label.** Everything here is *simulated*. Parameters are stipulated, not calibrated.
No claim about real payment data is made.

## 1 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

MASK = 0xFFFFFFFF
CRED_MAX, EPISODES, QUERY_FEE, FP_RATE, FEE_RATE = 12, 12, 5, 0.3, 0.01
BASE_RATE = 0.25          # fraud base rate
SEEDS = range(1, 201)     # 200 independent draws per point
print("numpy", np.__version__)

## 2 · The mechanism

`mulberry32` is ported from the JavaScript in the deployed game, including the operator
precedence of `+` over `^` and the short-circuit evaluation that makes a fraudulent episode
consume one more random number than a legitimate one. Porting it exactly is what lets Section 5
check the notebook against outputs recorded in the browser.

In [ ]:
def _imul(a, b):
    return ((a & MASK) * (b & MASK)) & MASK

def mulberry32(seed):
    a = seed & MASK
    def rnd():
        nonlocal a
        a = (a + 0x6D2B79F5) & MASK
        t = _imul(a ^ (a >> 15), 1 | a)
        t = (((t + _imul(t ^ (t >> 7), 61 | t)) & MASK) ^ t) & MASK
        return ((t ^ (t >> 14)) & MASK) / 4294967296.0
    return rnd

def draw_round(seed, round_idx, base, script_rate):
    """Twelve payment episodes, drawn exactly as the game draws them."""
    r = mulberry32((seed + round_idx * 7919) & MASK)
    eps = []
    for _ in range(EPISODES):
        fraud = r() < base
        scripted = (r() < script_rate) if fraud else False   # short-circuit, as in JS
        amount = round((200 + r() * 4800) / 10) * 10
        urgent = r() < (0.7 if fraud else 0.2)
        new_payee = r() < (0.85 if fraud else 0.35)
        r1, r2 = r(), r()
        eps.append(dict(fraud=fraud, scripted=scripted, amount=amount,
                        urgent=urgent, new_payee=new_payee, r1=r1, r2=r2))
    return eps

In [ ]:
def play_round(eps, mechanism, pool, credibility=CRED_MAX, habituation=True):
    """Apply one mechanism to every episode. The mechanism's signal decides; the player does not."""
    payoff, fraud_paid, legit_blocked, n_fraud = 0.0, 0, 0, 0
    for e in eps:
        n_fraud += e["fraud"]
        cost = 0.0
        if mechanism == "pay":
            pay = True
        elif mechanism == "hold":
            pay = False
        elif mechanism == "testimony":
            if credibility > 0:
                credibility -= 1
                attention = (credibility / CRED_MAX) if habituation else 1.0
                conflict = e["fraud"] and not e["scripted"] and e["r1"] < 0.75 * attention
                pay = not conflict
            else:
                pay = True                      # budget exhausted: the question is unavailable
        elif mechanism == "record":
            cost = QUERY_FEE
            flagged = (e["r2"] < pool) if e["fraud"] else (e["r2"] < 0.05)
            pay = not flagged
        else:
            raise ValueError(mechanism)

        d = -cost
        if pay and e["fraud"]:
            d -= e["amount"]; fraud_paid += 1
        if pay and not e["fraud"]:
            d += FEE_RATE * e["amount"]
        if (not pay) and (not e["fraud"]):
            d -= FP_RATE * e["amount"]; legit_blocked += 1
        payoff += d
    return dict(payoff=payoff, fraud_paid=fraud_paid, legit_blocked=legit_blocked,
                n_fraud=n_fraud, credibility=credibility)

def pooled_miss(mechanism, pool=0.0, script=0.5, habituation=True):
    """Miss rate pooled over seeds: fraud paid / fraud present. Normalising removes draw size."""
    paid = tot = 0
    for s in SEEDS:
        eps = draw_round(s, 0, BASE_RATE, script)
        r = play_round(eps, mechanism, pool, habituation=habituation)
        paid += r["fraud_paid"]; tot += r["n_fraud"]
    return paid / tot

## 3 · Run — sweep the reporting rate

The reporting rate is the share of fraudulent payees that other institutions have already
reported. It is an institutional parameter, not a property of the software: it measures
participation in pooled reporting.

In [ ]:
pool_grid = np.round(np.arange(0.0, 1.001, 0.05), 2)
miss_record = [pooled_miss("record", pool=p) for p in pool_grid]

fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.plot(pool_grid, 1 - pool_grid, ls="--", lw=1.2, color="0.6", label="predicted  1 − p")
ax.plot(pool_grid, miss_record, marker="o", ms=3.5, lw=1.6, color="#1f6f5c", label="simulated")
ax.set_xlabel("cross-institution reporting rate p")
ax.set_ylabel("miss rate (fraud paid / fraud present)")
ax.set_title("Record check: effectiveness is carried by participation")
ax.set_ylim(-0.02, 1.02); ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

print("max |simulated − predicted| =", round(max(abs(np.array(miss_record) - (1 - pool_grid))), 4))

## 4 · Change one assumption — is the failure habituation or authorship?

Two explanations compete for why testimony checks fail.

* **Habituation** (Anderson et al., 2016): repeated questioning depletes attention, so the
  customer stops noticing. The remedy would be an interface one.
* **Authorship**: the answer was written by the adversary, so it never conflicts regardless of
  attention. No interface change reaches this.

The ablation sets `habituation=False`, holding everything else fixed, and sweeps the script rate.
If the two curves converge as scripting rises, the second channel is doing the work at that end.

In [ ]:
script_grid = np.round(np.arange(0.0, 1.001, 0.05), 2)
miss_hab  = [pooled_miss("testimony", script=s, habituation=True)  for s in script_grid]
miss_nohab= [pooled_miss("testimony", script=s, habituation=False) for s in script_grid]

fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.plot(script_grid, miss_hab,   marker="o", ms=3.5, lw=1.6, color="#b26a00", label="with habituation (baseline)")
ax.plot(script_grid, miss_nohab, marker="s", ms=3.5, lw=1.6, color="#1f5fa8", label="habituation removed")
ax.fill_between(script_grid, miss_nohab, miss_hab, color="#b26a00", alpha=.12)
ax.set_xlabel("script rate (share of approaches that are coached)")
ax.set_ylabel("miss rate")
ax.set_title("Testimony check: two failure channels, only one is fixable by interface")
ax.set_ylim(-0.02, 1.02); ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

for s in [0.0, 0.5, 1.0]:
    i = list(script_grid).index(s)
    print(f"script {s:.2f}:  with habituation {miss_hab[i]:.3f}   without {miss_nohab[i]:.3f}"
          f"   gap {miss_hab[i]-miss_nohab[i]:.3f}")

## 5 · Check — does the notebook reproduce the deployed game?

The three figures below were recorded by hand in the browser on 2026-09-06 (seed 2026, base 0.25,
first round after a reset, every episode given the same mechanism). If the port is faithful, the
notebook reproduces them exactly.

In [ ]:
recorded = {("record", 0.0): -6775, ("testimony", 0.6): -4480, ("record", 0.6): -2005}

eps = draw_round(2026, 0, 0.25, 0.50)
print(f"fraudulent episodes in this draw: {sum(e['fraud'] for e in eps)} of {EPISODES}\n")
print(f"{'mechanism':<12}{'p':>5}{'notebook':>11}{'browser':>10}{'match':>8}")
ok = True
for (mech, pool), target in recorded.items():
    r = play_round(eps, mech, pool)
    same = round(r["payoff"]) == target
    ok &= same
    print(f"{mech:<12}{pool:>5}{round(r['payoff']):>11}{target:>10}{str(same):>8}")
print("\nAll three reproduced exactly." if ok else "\nMISMATCH — the port is not faithful.")

## 6 · Interpret

**What was obtained.** The record check's miss rate tracks `1 − p` across the whole sweep, so its
effectiveness is carried by participation in pooled reporting rather than by the software. The
testimony check degrades with habituation and *collapses* with authorship: removing habituation
helps at low script rates and does nothing at high ones, where both curves reach 1. The Python
port reproduces the three browser-recorded payoffs exactly, so the notebook and the game are the
same model rather than two similar ones.

**What this discriminates.** At high script rates the two explanations make different predictions
and the simulation separates them: habituation is a degradation an interface remedy could
address, authorship is a ceiling it cannot. That is the sense in which a mechanism reading
within-episode testimony cannot be made coaching-proof by improving the warning.

**What it does not establish.** Every number is simulated under stipulated parameters — the 0.05
false-flag rate, the 0.75 attention baseline, the 0.3 false-positive cost. The sweep confirms that
the model behaves as specified; it is not evidence about real payment systems, real customers, or
real reporting infrastructure. Calibration against payment-industry data and a behavioral estimate
of the habituation rate remain future work.

**What would change my mind.** If a testimony-based mechanism existed whose decision were
invariant to a passing script — for example one reading response latency or channel metadata
rather than content — H1 would be false as stated, and the criterion would need to be narrowed
from "testimony" to "testimony content".